# 02 · Probing the Atlas

Every page of **The Workstation Atlas** ends with a *"Probe the flow"* question. This notebook
turns them into Python you can actually run against this machine.

Notebook `01` drills the language. This one uses it — the questions are real, the answers come
from your filesystem, your `$PATH`, your git history.

**Ground rules.** Everything here is read-only: no writes outside a scratch directory in `/tmp`,
no network, no destructive git. Every helper degrades gracefully when a tool is missing, so the
notebook runs end to end even on a machine that looks nothing like this one.

> ⚠️ **This repo is public.** The outputs below describe your machine. A `nbstripout` clean filter
> is installed (see `.gitattributes`), so outputs are scrubbed on the way into git — you can run
> everything freely. Verify any time with `git diff --cached`.

In [ ]:
"""Run me first — helpers used by every probe below."""
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

HOME = Path.home()
CLONE_ROOT = HOME / "src" / "github.com"


def sh(*args: str, cwd: Path | str | None = None, timeout: int = 30) -> tuple[int, str, str]:
    """Run a command with no shell involved. Returns (returncode, stdout, stderr).

    Never raises: a missing binary comes back as rc=127, a hang as rc=124.
    """
    try:
        proc = subprocess.run(
            list(args), cwd=str(cwd) if cwd else None, capture_output=True,
            text=True, timeout=timeout, check=False,
        )
        return proc.returncode, proc.stdout.strip(), proc.stderr.strip()
    except FileNotFoundError:
        return 127, "", f"{args[0]}: not found"
    except subprocess.TimeoutExpired:
        return 124, "", f"{args[0]}: timed out after {timeout}s"


def git(*args: str, cwd: Path | str | None = None) -> tuple[int, str, str]:
    return sh("git", *args, cwd=cwd)


def table(rows: list[tuple], headers: tuple) -> None:
    """Minimal fixed-width table — no pandas needed for nine probes."""
    rows = [tuple(str(c) for c in r) for r in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in rows)) if rows else len(str(h))
              for i, h in enumerate(headers)]
    line = "  ".join(str(h).ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("  ".join("─" * w for w in widths))
    for r in rows:
        print("  ".join(c.ljust(w) for c, w in zip(r, widths)))


def repos_under(root: Path, limit: int = 200) -> list[Path]:
    """Every git working tree exactly one or two levels under `root`."""
    found = []
    if not root.is_dir():
        return found
    for org in sorted(root.iterdir()):
        # Skip compatibility symlinks — following them double-counts a repo under
        # both its old and new path (see ~/src/MOVELOG.md).
        if org.is_symlink() or not org.is_dir():
            continue
        if (org / ".git").exists():
            found.append(org)
            continue
        for repo in sorted(org.iterdir()):
            if repo.is_symlink() or not repo.is_dir():
                continue
            if (repo / ".git").exists():
                found.append(repo)
        if len(found) > limit:
            break
    return found


print(f"home       {HOME}")
print(f"clone root {CLONE_ROOT}  exists={CLONE_ROOT.is_dir()}")
rc, out, _ = git("--version")
print(f"git        {out or 'unavailable'}")

---
## ◎ · Master Map (atlas p.2)

> *"If `~/src` vanished, which column rebuilds it: the interfaces, or the wall?"*

The atlas's thesis is that Cursor, zsh and Claude Code never own state — GitHub does. That is a
falsifiable claim, and this is the falsification: **any repo with no remote is not on the other
side of the wall.** If `~/src` vanished, those are gone.

In [ ]:
rows, orphans, dirty = [], [], []

for repo in repos_under(CLONE_ROOT):
    rc, remote, _ = git("remote", "get-url", "origin", cwd=repo)
    has_remote = rc == 0 and remote
    rc2, status, _ = git("status", "--porcelain", cwd=repo)
    uncommitted = len([ln for ln in status.splitlines() if ln.strip()]) if rc2 == 0 else -1

    rel = repo.relative_to(CLONE_ROOT)
    if not has_remote:
        orphans.append(rel)
    if uncommitted > 0:
        dirty.append((rel, uncommitted))
    rows.append((str(rel)[:52], "yes" if has_remote else "NO", uncommitted if uncommitted >= 0 else "?"))

table(rows, ("repo", "remote?", "uncommitted"))
print()
print(f"{len(rows)} repos · {len(orphans)} with NO remote · {len(dirty)} with uncommitted work")
print()
print("Answer to the probe — the wall rebuilds everything EXCEPT these:")
for o in orphans:
    print(f"   ✗ {o}")
if not orphans:
    print("   (none — every repo here has a remote)")
print()
print("...and uncommitted work is not on the wall either, remote or not:")
for rel, n in sorted(dirty, key=lambda t: -t[1])[:10]:
    print(f"   ~ {rel}: {n} file(s)")

**Your answer.** _(double-click to edit)_

> The wall rebuilds …

---
## 1 · Course Overview + the Shell (atlas p.3)

> *"Run `which -a python3` — how many answers do you get, and which rung of your PATH decides?"*
> *"bun prepends itself after `~/.local/bin` in the file — so who actually wins a name collision, and why?"*

`$PATH` is a list, searched left to right, **first match wins**. `which -a` shows every match in
that order; `shutil.which` shows only the winner. The gap between the two is the answer.

In [ ]:
entries = os.environ.get("PATH", "").split(os.pathsep)

print("Your $PATH, in resolution order:")
for i, e in enumerate(entries, 1):
    marker = "←" if e.startswith(str(HOME)) else " "
    print(f"  {i:>2}. {e} {marker}")

print(f"\n{len(entries)} entries\n")

for name in ("python3", "python", "uv", "git", "node"):
    matches = []
    for idx, d in enumerate(entries, 1):
        candidate = Path(d) / name
        if candidate.is_file() and os.access(candidate, os.X_OK):
            matches.append((idx, str(candidate)))
    winner = shutil.which(name)
    print(f"{name}: {len(matches)} match(es); winner = {winner}")
    for idx, p in matches:
        flag = "  ← WINS (lowest PATH index)" if p == winner else ""
        print(f"       rung {idx:>2}: {p}{flag}")
    print()

In [ ]:
# The collision question, made concrete: which rung would win for a hypothetical name
# present in several directories?
from collections import defaultdict

seen = defaultdict(list)
for idx, d in enumerate(entries, 1):
    p = Path(d)
    if not p.is_dir():
        continue
    try:
        for f in p.iterdir():
            if f.is_file() and os.access(f, os.X_OK):
                seen[f.name].append(idx)
    except PermissionError:
        continue

collisions = {n: idxs for n, idxs in seen.items() if len(idxs) > 1}
print(f"{len(seen)} distinct executable names on PATH; {len(collisions)} appear in >1 directory\n")
print("Top collisions — the lowest rung number always wins:")
for name, idxs in sorted(collisions.items(), key=lambda kv: -len(kv[1]))[:12]:
    print(f"  {name:<22} rungs {sorted(idxs)}  → wins at rung {min(idxs)}: {shutil.which(name)}")

**Your answer.**

> `which -a python3` returns … and the deciding rung is … because …

---
## 2 · Command-line Environment (atlas p.4)

> *"Your health-poll loop retries 30× — what exit code does the function return if the server
> never comes up, and does anything downstream check it?"*

A shell function returns the exit code of its **last executed command** unless it `return`s
explicitly. That is the trap: a `for` loop that finishes without ever succeeding usually ends on
a *successful* `echo` or `sleep`, so the function reports **0 — success — after failing 30 times.**

In [ ]:
# Reproduce the trap without touching your real dotfiles.
poll_buggy = """
poll() {
  for i in 1 2 3; do
    false                     # stand-in for the failing curl
  done
  echo "gave up"              # ← last command succeeds, so the function returns 0
}
poll
"""

poll_fixed = """
poll() {
  for i in 1 2 3; do
    if false; then return 0; fi
  done
  echo "gave up" >&2
  return 1                    # ← explicit: the failure is reported
}
poll
"""

for label, script in (("buggy", poll_buggy), ("fixed", poll_fixed)):
    rc, out, err = sh("zsh", "-c", script)
    verdict = "reports SUCCESS after failing" if rc == 0 else "reports failure correctly"
    print(f"{label:<6} exit code = {rc}   → {verdict}")

print("\nAnd this is why it matters downstream — `&&` believes the exit code:")
rc, out, _ = sh("zsh", "-c", poll_buggy + "\n&& echo 'downstream ran anyway'")
print(f"  {out!r}")

In [ ]:
# Now look at your actual dotfiles: does any function poll without an explicit return?
zshrc = HOME / ".zshrc"
if not zshrc.exists():
    print("no ~/.zshrc on this machine")
else:
    text = zshrc.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()
    print(f"~/.zshrc: {len(lines)} lines, {zshrc.stat().st_size} bytes\n")

    in_fn, name, body = False, None, []
    findings = []
    for ln in lines:
        stripped = ln.strip()
        if not in_fn and ("() {" in stripped or stripped.startswith("function ")):
            in_fn, name, body = True, stripped.split("(")[0].replace("function ", "").strip(), []
            continue
        if in_fn:
            if stripped == "}":
                loops = any(w in "\n".join(body) for w in ("for ", "while ", "until "))
                explicit = any(b.strip().startswith("return") for b in body)
                if loops:
                    findings.append((name, len(body), "yes" if explicit else "NO ← trap"))
                in_fn = False
            else:
                body.append(ln)

    if findings:
        table(findings, ("function", "lines", "explicit return?"))
    else:
        print("no looping shell functions found in ~/.zshrc")

**Your answer.**

> The function returns … and downstream, … checks it.
>
> To make `jarvis-logs` survive a closed terminal, the lecture-2 primitive is …

---
## 3 · Development Environment & Tools (atlas p.5)

> *"Which of the four workspace folders gets the least L1 coverage (no LSP for its language)?"*

L1 is the language-server rung. Coverage means: is there a configured server for the language
this folder is actually written in? Count the source files, then look for the config that would
drive a server.

In [ ]:
import json as jsonlib

ws = HOME / "src" / "workspaces" / "ai-architect.code-workspace"
folders = []
if ws.exists():
    raw = ws.read_text(encoding="utf-8", errors="replace")
    # workspace files permit // comments, which strict JSON does not
    cleaned = "\n".join(ln for ln in raw.splitlines() if not ln.strip().startswith("//"))
    try:
        folders = [f.get("path") for f in jsonlib.loads(cleaned).get("folders", [])]
    except jsonlib.JSONDecodeError as exc:
        print(f"could not parse workspace file: {exc}")
else:
    print(f"no workspace file at {ws}")

EXT = {".py": "python", ".rs": "rust", ".ts": "typescript", ".tsx": "typescript",
       ".js": "javascript", ".go": "go", ".dart": "dart", ".ipynb": "python"}
L1_CONFIG = {"python": ("pyproject.toml", "ruff.toml", ".ruff.toml", "setup.cfg", "mypy.ini"),
             "rust": ("Cargo.toml",), "typescript": ("tsconfig.json",),
             "javascript": ("jsconfig.json", "package.json"), "go": ("go.mod",),
             "dart": ("pubspec.yaml",)}

rows = []
for entry in folders:
    p = Path(os.path.expandvars(str(entry))).expanduser()
    if not p.is_absolute():
        p = (ws.parent / p).resolve()
    if not p.is_dir():
        rows.append((str(entry)[:34], "—", "MISSING FOLDER", "—"))
        continue

    counts = {}
    for f in p.rglob("*"):
        if any(part in {".git", ".venv", "node_modules", "target", "__pycache__"} for part in f.parts):
            continue
        lang = EXT.get(f.suffix)
        if lang:
            counts[lang] = counts.get(lang, 0) + 1
    if not counts:
        rows.append((p.name[:34], "—", "no source found", "—"))
        continue

    primary = max(counts, key=counts.get)
    cfgs = [c for c in L1_CONFIG.get(primary, ()) if (p / c).exists()]
    rows.append((p.name[:34], f"{primary} ({counts[primary]})",
                 ", ".join(cfgs) if cfgs else "NONE at root",
                 "✓" if cfgs else "✗ least covered"))

table(rows, ("workspace folder", "primary lang (files)", "L1 config at root", "L1?"))

**Your answer.**

> The thinnest L1 coverage is … because …

---
## 4 · Debugging & Profiling (atlas p.6)

> *"What's the real/user/sys split of your slowest test suite — and have you ever measured it?"*

**real** is wall-clock. **user** is CPU burned in your code. **sys** is CPU burned inside the
kernel on your behalf. The ratios diagnose the bottleneck before you optimise anything:

| Shape | Reading |
|---|---|
| real ≫ user + sys | waiting — I/O, network, sleep. Optimising the code will not help. |
| user ≫ sys | genuinely CPU-bound in your code. Profile it. |
| sys high | syscall-heavy — file churn, process spawning. |

In [ ]:
import resource
import time


def timed_run(*args: str, cwd=None) -> dict:
    """real/user/sys for a child process, straight from getrusage."""
    before = resource.getrusage(resource.RUSAGE_CHILDREN)
    t0 = time.perf_counter()
    rc, out, err = sh(*args, cwd=cwd, timeout=120)
    real = time.perf_counter() - t0
    after = resource.getrusage(resource.RUSAGE_CHILDREN)
    user = after.ru_utime - before.ru_utime
    sys_ = after.ru_stime - before.ru_stime
    return {"cmd": " ".join(args), "rc": rc, "real": real, "user": user, "sys": sys_,
            "out": out, "err": err}


def report(m: dict) -> None:
    busy = m["user"] + m["sys"]
    shape = ("waiting on I/O — code optimisation will not help" if m["real"] > 2 * busy and busy >= 0
             else "CPU-bound in userland — profile it" if m["user"] > 3 * max(m["sys"], 1e-9)
             else "syscall-heavy")
    print(f"  {m['cmd'][:58]:<58} rc={m['rc']}")
    print(f"    real {m['real']:6.3f}s   user {m['user']:6.3f}s   sys {m['sys']:6.3f}s")
    print(f"    → {shape}\n")


print("Three deliberately different shapes:\n")
report(timed_run("python3", "-c", "import time; time.sleep(0.4)"))            # pure waiting
report(timed_run("python3", "-c", "sum(i*i for i in range(4_000_000))"))      # pure CPU
report(timed_run("python3", "-c",
                 "import os;[os.stat('/usr/bin/env') for _ in range(60_000)]"))  # syscalls

In [ ]:
# Point it at something real. Edit CANDIDATES to a suite you care about.
CANDIDATES = [
    (CLONE_ROOT / "JeremyGracey-AI" / "anthropic-swe-prep", ["python3", "-m", "pytest", "-q", "--collect-only"]),
]

ran = False
for repo, cmd in CANDIDATES:
    if not repo.is_dir():
        print(f"skip (not present): {repo}")
        continue
    ran = True
    print(f"\n{repo.name}:")
    report(timed_run(*cmd, cwd=repo))

if not ran:
    print("\nNo candidate suites found — add one above and re-run.")
print("Now answer honestly: had you ever measured that split before this cell?")

**Your answer.**

> real/user/sys for my slowest suite is … / … / …, which means the bottleneck is …

---
## 5 · Version Control & Git (atlas p.7)

> *"Without running it: after a rebase of a 3-commit branch, how many new commit objects exist,
> and what happened to the old three?"*

Commit yourself to an answer before running the next cell. Then run it — it builds a throwaway
repo in `/tmp`, does the rebase, and counts the objects.

In [ ]:
import tempfile

lab = Path(tempfile.mkdtemp(prefix="atlas-rebase-"))
G = ("-c", "user.email=probe@example.com", "-c", "user.name=Probe",
     "-c", "commit.gpgsign=false", "-c", "init.defaultBranch=main")


def commit(msg: str, repo: Path) -> None:
    (repo / f"{msg}.txt").write_text(msg, encoding="utf-8")
    git(*G, "add", "-A", cwd=repo)
    git(*G, "commit", "-q", "-m", msg, cwd=repo)


def commit_objects(repo: Path) -> set[str]:
    rc, out, _ = git("cat-file", "--batch-all-objects", "--batch-check=%(objectname) %(objecttype)", cwd=repo)
    return {ln.split()[0] for ln in out.splitlines() if ln.endswith(" commit")}


git(*G, "init", "-q", str(lab))
commit("base", lab)
git(*G, "checkout", "-q", "-b", "feature", cwd=lab)
for m in ("one", "two", "three"):
    commit(m, lab)
git(*G, "checkout", "-q", "main", cwd=lab)
commit("upstream-moved", lab)

before = commit_objects(lab)
rc, sha_before, _ = git("rev-list", "feature", "--max-count=3", cwd=lab)
print(f"commit objects BEFORE rebase: {len(before)}")
print("feature tips before:", sha_before.split()[0][:8] if sha_before else "?")

git(*G, "checkout", "-q", "feature", cwd=lab)
rc, out, err = git(*G, "rebase", "main", cwd=lab)
print(f"\nrebase rc={rc} {err.splitlines()[0] if err else ''}")

after = commit_objects(lab)
new = after - before
still_there = before & after

print(f"\ncommit objects AFTER rebase:  {len(after)}")
print(f"  brand-new commit objects:   {len(new)}   ← the answer")
print(f"  original objects still present: {len(still_there)} of {len(before)}")

rc, reachable, _ = git("rev-list", "--all", "--count", cwd=lab)
print(f"  reachable from a branch:    {reachable}")

# Two different questions, and the flags matter:
#   plain `fsck --unreachable` counts the REFLOG as a reference, so the old commits
#   look reachable. `--no-reflogs` asks the sharper question: what is kept alive by
#   nothing but the reflog?
rc, with_reflog, _ = git("fsck", "--unreachable", "--no-progress", cwd=lab)
rc, without_reflog, _ = git("fsck", "--unreachable", "--no-reflogs", "--no-progress", cwd=lab)
n_with = len([ln for ln in with_reflog.splitlines() if "unreachable commit" in ln])
n_without = len([ln for ln in without_reflog.splitlines() if "unreachable commit" in ln])

print(f"  unreachable, reflog counted:   {n_with}   ← reflog still points at the originals")
print(f"  unreachable, reflog ignored:   {n_without}   ← the old three, orphaned but NOT deleted")
print(f"\n  arithmetic: {len(before)} before + {len(new)} new = {len(after)} objects; "
      f"{reachable} on a branch, {n_without} reachable only via reflog")
print("\nSo: rebase COPIES. The originals survive in the object store, held by the reflog,")
print("until `git gc` prunes them — which is exactly what `git reflog` recovery relies on.")

In [ ]:
# "Which of your org roots would git bisect help most — and is its test fast enough to bisect with?"
# bisect does log2(N) checkouts; a suite that takes T seconds costs about T * log2(N).
import math

rows = []
for repo in repos_under(CLONE_ROOT):
    rc, count, _ = git("rev-list", "--count", "HEAD", cwd=repo)
    if rc != 0 or not count.isdigit():
        continue
    n = int(count)
    if n < 20:
        continue
    has_tests = any((repo / d).is_dir() for d in ("tests", "test")) or \
                any(repo.glob("**/test_*.py")) or any(repo.glob("**/*_test.rs"))
    steps = math.ceil(math.log2(n)) if n > 1 else 1
    rows.append((str(repo.relative_to(CLONE_ROOT))[:46], n, steps, "yes" if has_tests else "NO"))

rows.sort(key=lambda r: -r[1])
table(rows[:15], ("repo", "commits", "bisect steps", "tests?"))
print("\nbisect pays off where commits are many AND a fast test exists.")
print("A repo with many commits and no test is where bisect would help most — and cannot.")

**Your answer.**

> New commit objects: … . The old three: … .

---
## 6 · Packaging & Shipping Code (atlas p.8)

> *"For each of your three ladders — uv, cargo, bun — where is the lockfile, and is it committed?
> (One of them may surprise you.)"*

S4 on the ladder is **Lock**. An uncommitted lockfile is not a lock — it is a local convenience
that reproduces nothing for anyone else.

In [ ]:
LOCKS = {"uv.lock": "uv (Python)", "poetry.lock": "poetry (Python)",
         "Cargo.lock": "cargo (Rust)", "bun.lockb": "bun (TS/JS)", "bun.lock": "bun (TS/JS)",
         "package-lock.json": "npm (TS/JS)", "pnpm-lock.yaml": "pnpm (TS/JS)"}

rows = []
for repo in repos_under(CLONE_ROOT):
    for lockname, ladder in LOCKS.items():
        for lock in list(repo.rglob(lockname))[:6]:
            if any(part in {".venv", "node_modules", "target", ".git"} for part in lock.parts):
                continue
            rel_in_repo = lock.relative_to(repo)
            rc, out, _ = git("ls-files", "--error-unmatch", str(rel_in_repo), cwd=repo)
            tracked = rc == 0
            rc2, ign, _ = git("check-ignore", str(rel_in_repo), cwd=repo)
            ignored = rc2 == 0
            state = "committed" if tracked else ("GITIGNORED ←" if ignored else "untracked ←")
            rows.append((ladder, str(repo.relative_to(CLONE_ROOT))[:36], str(rel_in_repo)[:30], state))

rows.sort(key=lambda r: (r[0], r[3] == "committed", r[1]))
table(rows, ("ladder", "repo", "lockfile", "S4 state"))

bad = [r for r in rows if r[3] != "committed"]
print(f"\n{len(rows)} lockfiles · {len(bad)} not committed")
for r in bad:
    print(f"   ⚠ {r[0]}: {r[1]}/{r[2]} is {r[3].rstrip(' ←')}")
if not bad:
    print("   every lockfile found is committed — S4 holds across all ladders")

**Your answer.**

> uv → … · cargo → … · bun → … . The surprise was …

---
## 7 · Agentic Coding (atlas p.9)

> *"For your last agent-written change: which D-rung caught the mistake — or would have, if
> there'd been one?"*

The D-stack from the atlas: **D1** context curation · **D2** isolation · **D3** process gates ·
**D4** governance. Inventory what is actually installed, then judge which rung was load-bearing.

In [ ]:
CLAUDE_DIR = HOME / ".claude"

def count_glob(root: Path, pattern: str) -> int:
    return len(list(root.glob(pattern))) if root.is_dir() else 0

inventory = [
    ("D1", "CLAUDE.md files (home + repos)",
     (1 if (HOME / "CLAUDE.md").exists() else 0)
     + sum(1 for r in repos_under(CLONE_ROOT) if (r / "CLAUDE.md").exists())),
    ("D1", "skills installed", count_glob(CLAUDE_DIR, "skills/*") + count_glob(CLAUDE_DIR, "plugins/cache/*/*/*/skills/*")),
    ("D1", "memory files", count_glob(CLAUDE_DIR, "projects/*/memory/*.md")),
    ("D2", "git worktrees registered", sum(
        1 for r in repos_under(CLONE_ROOT)
        for ln in git("worktree", "list", cwd=r)[1].splitlines()[1:])),
    ("D3", "plugins cached", count_glob(CLAUDE_DIR, "plugins/cache/*/*")),
    ("D3", "hooks / settings files", count_glob(CLAUDE_DIR, "settings*.json")),
    ("D4", "repos with a CI workflow", sum(
        1 for r in repos_under(CLONE_ROOT) if (r / ".github" / "workflows").is_dir())),
]
table([(d, label, n) for d, label, n in inventory], ("rung", "what", "count"))

print("\nRepos carrying their own CLAUDE.md (D1 — curated context, per-project):")
for r in repos_under(CLONE_ROOT):
    if (r / "CLAUDE.md").exists():
        print(f"   · {r.relative_to(CLONE_ROOT)}")

**Your answer.**

> The rung that caught it was … . The repo where I'd still forbid autonomous edits is … , which
> says its tests are … .

---
## 8 · Beyond the Code (atlas p.10)

> *"Open your last five commit messages: do any explain **why**? Would a stranger know what to
> revert?"*

The mechanical proxy: a subject line says *what*; a body says *why*. A commit with no body has
nowhere to have explained itself.

In [ ]:
SAMPLE = 5
rows = []
for repo in repos_under(CLONE_ROOT):
    rc, out, _ = git("log", f"-{SAMPLE}", "--format=%h%x00%s%x00%b%x00%an", cwd=repo)
    if rc != 0 or not out:
        continue
    with_body = 0
    subjects = []
    for entry in out.split("\n"):
        parts = entry.split("\x00")
        if len(parts) < 4:
            continue
        sha, subject, body, author = parts[0], parts[1], parts[2], parts[3]
        if body.strip():
            with_body += 1
        subjects.append(subject)
    if subjects:
        conventional = sum(1 for s in subjects if ":" in s.split(" ")[0])
        rows.append((str(repo.relative_to(CLONE_ROOT))[:40], len(subjects),
                     f"{with_body}/{len(subjects)}", f"{conventional}/{len(subjects)}"))

rows.sort(key=lambda r: int(r[2].split("/")[0]))
table(rows, ("repo", "sampled", "explain WHY (has body)", "conventional prefix"))

worst = [r for r in rows if r[2].startswith("0/")]
print(f"\n{len(worst)} repo(s) where NOT ONE of the last {SAMPLE} commits has a body:")
for r in worst[:12]:
    print(f"   ✗ {r[0]}")

In [ ]:
# Read the actual messages from the repo you care about most.
TARGET = CLONE_ROOT / "JeremyGracey-AI" / "The-Workstation-Atlas"

if TARGET.is_dir():
    rc, out, _ = git("log", "-5", "--format=%C(auto)%h %s%n    %b", cwd=TARGET)
    print(f"{TARGET.name}:\n")
    print(out or "(no commits yet)")
else:
    print(f"not found: {TARGET}")

**Your answer.**

> Of my last five, … explain why. A stranger would/would not know what to revert because …

---
## 9 · Code Quality (atlas p.11)

> *"The funnel only works if every layer runs on every change. Which of your active repos skips a
> layer, and what class of bug does that admit?"*

Format → Lint → Test → CI. Each layer catches a class the one above cannot. A missing layer is not
a style preference; it is a named category of bug you have agreed to ship.

In [ ]:
LAYERS = {
    "format/lint": ("pyproject.toml", "ruff.toml", ".ruff.toml", ".prettierrc", "rustfmt.toml", ".editorconfig"),
    "test":        ("tests", "test", "pytest.ini", "tox.ini", "conftest.py"),
    "pre-commit":  (".pre-commit-config.yaml", ".husky"),
    "CI":          (".github/workflows",),
}
ADMITS = {"format/lint": "style drift + the pattern-level bugs a linter catches (unused vars, shadowing, unsafe calls)",
          "test": "behavioural regressions — nothing proves the change did what it claimed",
          "pre-commit": "broken code reaching the remote; the funnel becomes opt-in",
          "CI": "'works on my machine' — no layer runs on anyone else's change"}

rows, gaps = [], {k: [] for k in LAYERS}
for repo in repos_under(CLONE_ROOT):
    rc, count, _ = git("rev-list", "--count", "HEAD", cwd=repo)
    if rc != 0 or not count.isdigit() or int(count) < 3:
        continue                                   # skip near-empty repos
    marks = []
    for layer, markers in LAYERS.items():
        present = any((repo / m).exists() for m in markers)
        marks.append("✓" if present else "·")
        if not present:
            gaps[layer].append(repo.relative_to(CLONE_ROOT))
    rows.append((str(repo.relative_to(CLONE_ROOT))[:40], count, *marks))

rows.sort(key=lambda r: sum(1 for c in r[2:] if c == "✓"))
table(rows, ("repo", "commits", *LAYERS.keys()))

print("\nWhat each gap admits:\n")
for layer, missing in gaps.items():
    if missing:
        print(f"  ✗ {layer}  — missing in {len(missing)} repo(s)")
        print(f"      admits: {ADMITS[layer]}")
        print(f"      e.g. {', '.join(str(m) for m in missing[:4])}\n")

**Your answer.**

> The repo that skips the most is … , which admits …

---
## Closing the loop

The atlas asserts things about this desk. This notebook is where those assertions get checked
against the filesystem — which is the *Evaluated* pillar from `~/CLAUDE.md` applied to the atlas
itself: **a claim with a stated pass condition, and evidence.**

When a probe's answer surprises you, that is a finding. Two ways to record it:

1. Edit the relevant page in `atlas/missing-semester-workstation-atlas.html` — it is the master
   copy now, under version control.
2. Write it down here in the *Your answer* cells and commit. Outputs are stripped by the
   `nbstripout` filter, but your prose in markdown cells is kept.

In [ ]:
print("Probes complete. Nothing above wrote outside /tmp.")
print("\nVerify the stripping filter is doing its job before you push:")
print("   git add notebooks/ && git diff --cached --stat && git diff --cached | grep -c '\"output_type\"'")
print("   (that grep should print 0)")